# Chorus -- opinion-CSV echo chamber experiment (Colab)

Runs **only** the opinion-CSV pipeline (`sandbox/opinion_*.py`,
`scripts/run_opinion_experiment.py`) -- agents write free-text opinions to
`initial_stance.csv`, self-report `BELIEF`/`CONFIDENCE` each round, sample
neighbours filtered by belief similarity, and `opinion_round_metrics.csv`
tracks Polarization Index, Global Disagreement, NCI and mean belief shift
per round. This notebook does **not** run the RQ1 bank/generated campaign
(`run_simulation.py`) -- that is a separate pipeline.

Inference is self-hosted via vLLM on the Colab GPU, so this costs GPU time
and nothing else.

**Two hazards**, same ones the RQ1 notebook documents:

1. `asyncio.run()` raises inside a notebook cell, so every actual run is
   launched as a subprocess (`!python scripts/run_opinion_experiment.py ...`),
   never called as a plain Python function from a cell.
2. Colab's filesystem is ephemeral -- mount Drive and point `OUTPUT_DIR`
   at it (Cell 2) if you want `initial_stance.csv` / `opinion_round_metrics.csv`
   to survive the runtime disconnecting.

## 1. Mount Drive (optional, but recommended for persistent output)

In [ ]:
MOUNT_DRIVE = True  # set False to keep everything on the ephemeral local disk

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/chorus_opinion_runs"
else:
    DRIVE_ROOT = "/content/chorus_opinion_runs"

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("output will be written under:", DRIVE_ROOT)

## 2. Get the repo

Clones via HTTPS. If the repo is private, upload a zip instead and unzip it to `/content/Chorus` -- either way this notebook only needs `%cd` to land inside the checkout before Cell 3.

In [ ]:
REPO_URL = "https://github.com/ResearchDrafts/Chorus--Agentic-Network-Orchestration-and-Analysis.git"
REPO_DIR = "/content/Chorus"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 3. Install dependencies

In [ ]:
!pip install -q -e ".[dev]"
!pip install -q vllm

## 4. Serve the model with vLLM

One combined cell: cleanup, a GPU-free check, launch, and the wait loop. Every repeat failure in this notebook traced back to cleanup and launch being split across cells, so it was never clear whether a leftover process was actually gone before the next launch tried to use the GPU -- this cell checks that explicitly instead of assuming it.

In [ ]:
import subprocess, sys, time, urllib.request, urllib.error

MODEL = "Qwen/Qwen2.5-7B-Instruct"
PORT = 8000
API_BASE = f"http://localhost:{PORT}/v1"
MODEL_BACKEND_ID = f"hosted_vllm/{MODEL}"


def _gpu_used_mb():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    return int(result.stdout.strip().split("\n")[0])


def _tail_log(n=40):
    try:
        with open("/content/vllm_server.log") as f:
            return "".join(f.readlines()[-n:])
    except FileNotFoundError:
        return "(no log file found)"


# 1. Kill any leftover vLLM process from a previous attempt. start_new_session
# (below) detaches vLLM from the kernel's process group, which also means
# Restart session does NOT kill it -- it survives as an orphan holding GPU
# memory, and the next launch fails trying to grab memory never given back.
subprocess.run(["pkill", "-9", "-f", "vllm"])

# 2. Confirm the GPU is ACTUALLY free before launching anything -- killing a
# process and the driver reclaiming its memory are not instantaneous, and
# three separate failures in this debugging session trace back to launching
# into a GPU that still had a dead run's memory on it.
for attempt in range(15):
    used = _gpu_used_mb()
    if used < 500:
        break
    print(f"waiting for GPU memory to clear... ({used} MiB still in use)")
    time.sleep(2)
else:
    raise RuntimeError(
        f"GPU still shows {_gpu_used_mb()} MiB in use after killing vllm "
        "processes. pkill isn't enough this time -- go to Runtime > "
        "Disconnect and delete runtime (not just Restart session; that "
        "only restarts the kernel, not the underlying VM) and start over "
        "from Cell 1 on a guaranteed-clean VM."
    )

# 3. Launch. -u forces unbuffered output (otherwise a real crash's traceback
# can be lost in an unflushed buffer); --max-num-seqs matches max_concurrency
# in the run config so vLLM's own batching capacity is never the bottleneck;
# start_new_session isolates it from notebook-level interrupts.
vllm_log = open("/content/vllm_server.log", "w")
vllm_proc = subprocess.Popen(
    [sys.executable, "-u", "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--dtype", "half", "--port", str(PORT),
     "--max-num-seqs", "32", "--timeout-keep-alive", "120"],
    stdout=vllm_log, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("vLLM server starting, pid:", vllm_proc.pid)

# 4. Wait, checking the subprocess itself every iteration -- a server that
# already crashed must never look identical to one still loading.
start = time.time()
deadline = start + 600
url = f"{API_BASE}/models"
while time.time() < deadline:
    exit_code = vllm_proc.poll()
    if exit_code is not None:
        raise RuntimeError(
            f"vLLM server process exited early (code {exit_code}) after "
            f"{time.time() - start:.0f}s. Last lines of /content/vllm_server.log:\n\n"
            f"{_tail_log()}"
        )
    try:
        with urllib.request.urlopen(url, timeout=5) as resp:
            body = resp.read().decode("utf-8", "replace")
        if MODEL in body:
            print(f"server is up and serving {MODEL} (after {time.time() - start:.0f}s)")
            break
    except urllib.error.URLError:
        pass
    print(f"waiting for vLLM server... ({time.time() - start:.0f}s elapsed)")
    time.sleep(5)
else:
    raise TimeoutError(
        f"vLLM server did not come up within 600s -- "
        f"check /content/vllm_server.log. Last lines:\n\n{_tail_log()}"
    )

## 5. Configure the run

Edit the variables below directly. `LANGUAGES` follows plan.md's rule:
exactly one `True` forces every agent to that language; all four `True`
mixes the population by each persona's own `language preference` in
`data/personas/pool_100.jsonl`; anything else (2 or 3 `True`) is
deliberately unsupported and `run_opinion_experiment.py` will refuse to
start.

In [ ]:
import yaml, os

RUN_ID = "opinion_colab_run2_k30"  # NEW id -- opinion_colab_run1 holds your completed K=5 dataset, do not reuse it
TOPIC = "Is there such a thing as a paradox of free speech?"

M = 100     # null (None) uses every persona in the pool
K = 30      # number of update rounds

LANGUAGES = {
    "english": True,
    "hindi": False,
    "hinglish": False,
    "romanized_hindi": False,
}

SIMILARITY_THRESHOLD = 2.0   # belief/v scale, [-2, 2] -- not the x=[-1,1] scale
# N must be strictly LESS than GRAPH_DEGREE, not just <=. Watts-Strogatz
# rewiring moves edges rather than adding them, so N == GRAPH_DEGREE leaves
# zero slack -- any node that loses even one edge to rewiring (graph_rewire_p)
# then has fewer than N neighbours and GraphSampling refuses to start. N=5
# with degree=6 mirrors this repo's own e8/e9 configs, which use exactly
# this gap for the same reason.
N = 5                        # neighbours sampled per round
GRAPH_DEGREE = 6             # even, and < M; keep at least 1 more than N
GRAPH_REWIRE_P = 0.1
SEED = 1000
# Even with 1 slack (degree = N + 1), Watts-Strogatz rewiring strands a few
# nodes below N neighbours most of the time at M=100 -- this is measured,
# not bad luck with this particular seed. True delivers a shorter feed to
# just those nodes instead of refusing to start, mirroring this repo's own
# e9 config, which opts into the same thing for the same reason.
ALLOW_SHORT_FEED = True

OUTPUT_DIR = f"{DRIVE_ROOT}/{RUN_ID}"

config = {
    "run_id": RUN_ID,
    "topic": TOPIC,
    "M": M,
    "K": K,
    "languages": LANGUAGES,
    "similarity_threshold": SIMILARITY_THRESHOLD,
    "N": N,
    "graph_degree": GRAPH_DEGREE,
    "graph_rewire_p": GRAPH_REWIRE_P,
    "allow_short_feed": ALLOW_SHORT_FEED,
    "seed": SEED,
    "model_backend_id": MODEL_BACKEND_ID,
    "api_base": API_BASE,
    "persona_pool_path": "data/personas/pool_100.jsonl",
    "max_reply_tokens": 512,
    "max_concurrency": 32,
    "output_dir": OUTPUT_DIR,
}

CONFIG_PATH = f"/content/{RUN_ID}.yaml"
with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(open(CONFIG_PATH).read())

### Patches for this session

Two fixes that only exist in the local repo copy this notebook was built from, not yet in what `git clone` pulled from GitHub. Both are idempotent (safe to re-run, and a no-op once the real fix is pushed and re-cloned).

In [ ]:
# Patch 1: allow_short_feed. Watts-Strogatz rewiring strands a few nodes
# below N neighbours most of the time at M=100 even with 1 slack
# (graph_degree = N + 1) -- measured, not bad luck with one seed. This field
# lets GraphSampling deliver a shorter feed to just those nodes instead of
# refusing to start, mirroring this repo's own e9 config's same escape hatch.
models_path = "sandbox/opinion_models.py"
with open(models_path) as f:
    content = f.read()

if "allow_short_feed" not in content:
    marker = "graph_rewire_p: float = Field(ge=0.0, le=1.0)"
    if marker not in content:
        raise RuntimeError(f"marker not found in {models_path} -- the file changed shape, paste its current content back for a new patch")
    content = content.replace(marker, marker + "\n    allow_short_feed: bool = False", 1)
    with open(models_path, "w") as f:
        f.write(content)
    print("patched", models_path)
else:
    print(models_path, "already has allow_short_feed")

orch_path = "sandbox/opinion_orchestrator.py"
with open(orch_path) as f:
    content = f.read()

if "allow_short_feed=config.allow_short_feed" not in content:
    old_call = (
        "graph = GraphSampling(\n"
        "        agent_ids, config.graph_degree, config.graph_rewire_p, config.N, seed_mgr.graph_seed\n"
        "    )"
    )
    if old_call not in content:
        raise RuntimeError(f"marker not found in {orch_path} -- the file changed shape, paste its current content back for a new patch")
    new_call = (
        "graph = GraphSampling(\n"
        "        agent_ids, config.graph_degree, config.graph_rewire_p, config.N, seed_mgr.graph_seed,\n"
        "        allow_short_feed=config.allow_short_feed,\n"
        "    )"
    )
    content = content.replace(old_call, new_call, 1)
    with open(orch_path, "w") as f:
        f.write(content)
    print("patched", orch_path)
else:
    print(orch_path, "already wires allow_short_feed through")

In [ ]:
# Patch 2: export_csv.py lives at the repo root, not inside the installed
# sandbox/analysis packages. analysis/metrics.py imports it as a bare
# top-level module, which only resolves if the repo root is on sys.path --
# true by accident in some local dev envs, not guaranteed in a fresh
# editable install. This sets PYTHONPATH for every subsequent !python call
# in this session.
import os
os.environ["PYTHONPATH"] = REPO_DIR
print("PYTHONPATH set to", REPO_DIR)

## 6. Dry run

Validates the config, resolves the language toggle, and confirms the
persona pool parses -- without dispatching a single call. Catches a
partial language-toggle combination, an `M` larger than the pool, or a
malformed persona line before anything is billed.

In [ ]:
!python scripts/run_opinion_experiment.py {CONFIG_PATH} --dry-run

## 7. Run it

Launched as a subprocess (`!python ...`), per Hazard 1 above -- `asyncio.run()`
inside `scripts/run_opinion_experiment.py` cannot be called directly from a
notebook cell. No resume/checkpoint support: if this cell is interrupted,
re-running it starts the whole run over (give it a new `RUN_ID` first if you
don't want to overwrite the partial output).

In [ ]:
!python scripts/run_opinion_experiment.py {CONFIG_PATH}

## 8. Load the results

In [ ]:
import pandas as pd

initial_stance = pd.read_csv(f"{OUTPUT_DIR}/initial_stance.csv")
round_metrics = pd.read_csv(f"{OUTPUT_DIR}/opinion_round_metrics.csv")

print(initial_stance.shape, "agents x columns")
initial_stance.head()

In [ ]:
round_metrics

## 9. Plot Pz / DG / NCI / mean belief shift over rounds

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].plot(round_metrics["round"], round_metrics["polarization_index"], marker="o")
axes[0, 0].set_title("Polarization Index (Pz)")

axes[0, 1].plot(round_metrics["round"], round_metrics["global_disagreement"], marker="o")
axes[0, 1].set_title("Global Disagreement (DG)")

axes[1, 0].plot(round_metrics["round"], round_metrics["nci"], marker="o")
axes[1, 0].set_title("Neighbor Correlation Index (NCI)")

axes[1, 1].plot(round_metrics["round"], round_metrics["mean_belief_shift"], marker="o", label="mean (signed)")
axes[1, 1].plot(round_metrics["round"], round_metrics["mean_abs_individual_shift"], marker="o", label="mean |shift|")
axes[1, 1].set_title("Mean Belief Shift")
axes[1, 1].legend()

for ax in axes.flat:
    ax.set_xlabel("round")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Comparing language conditions

Rerun Cell 5 onward with a different `RUN_ID` and a different `LANGUAGES`
toggle (same `SEED`, same everything else) to get a comparable arm --
`M`/`K`/`seed`/graph settings held identical is what makes two runs'
`opinion_round_metrics.csv` comparable to each other. Each run is a full
end-to-end pass (no resume), so budget GPU time accordingly: roughly
`M * (K + 1)` LLM calls per run.

Remember to `vllm_proc.terminate()` when you are done serving, if you plan
to reuse the GPU for something else in the same session.